In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

In [55]:
yellow_keep = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount"
]


# inspect jan 2024 first 
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

df = pd.read_parquet(url, columns=yellow_keep)

print("Working shape:", df.shape)
print("Approx memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
display(df.head())

Working shape: (2964624, 7)
Approx memory usage (MB): 135.71


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,passenger_count,fare_amount
0,2024-01-01 00:57:55,2024-01-01 01:17:43,186,79,1.72,1.0,17.7
1,2024-01-01 00:03:00,2024-01-01 00:09:36,140,236,1.80,1.0,10.0
2,2024-01-01 00:17:06,2024-01-01 00:35:01,236,79,4.70,1.0,23.3
3,2024-01-01 00:36:38,2024-01-01 00:44:56,79,211,1.40,1.0,10.0
4,2024-01-01 00:46:51,2024-01-01 00:52:57,211,148,0.80,1.0,7.9


In [56]:

# 1. Initial inspection
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)
display(missing_summary)

display(df["passenger_count"].value_counts(dropna=False).sort_index().to_frame("count"))
display(df["trip_distance"].describe().to_frame("trip_distance"))
display(df["fare_amount"].describe().to_frame("fare_amount"))

,missing_count
passenger_count,140162
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_distance,0
fare_amount,0


,count
passenger_count,
0.0,31465
1.0,2188739
2.0,405103
3.0,91262
4.0,51974
5.0,33506
6.0,22353
7.0,8
8.0,51


,trip_distance
count,2.964624e+06
mean,3.652169e+00
std,2.254626e+02
min,0.000000e+00
25%,1.000000e+00
50%,1.680000e+00
75%,3.110000e+00
max,3.127223e+05


,fare_amount
count,2.964624e+06
mean,1.817506e+01
std,1.894955e+01
min,-8.990000e+02
25%,8.600000e+00
50%,1.280000e+01
75%,2.050000e+01
max,5.000000e+03


In [57]:

# 2. Type screening
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

display(pd.DataFrame(df.dtypes, columns=["dtype"]))

,dtype
tpep_pickup_datetime,datetime64[us]
tpep_dropoff_datetime,datetime64[us]
PULocationID,int32
DOLocationID,int32
trip_distance,float64
passenger_count,float64
fare_amount,float64


In [58]:

# 3. Minimal early cleaning
# simple filters first to shrink the data
print("Rows before early cleaning:", len(df))

# must-have fields for demand analysis
df = df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
])

# validity filters
df = df[df["fare_amount"] > 0]
df = df[df["trip_distance"] > 0.2]
df = df[df["trip_distance"] <= 120]
df = df[df["PULocationID"].between(1, 263)]
df = df[df["DOLocationID"].between(1, 263)]

# passenger_count used as validation only
df = df[df["passenger_count"].notna()]
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 5)]

print("Rows after early cleaning:", len(df))

Rows before early cleaning: 2964624
Rows after early cleaning: 2660242


In [59]:

# 4. Create validation features
duration_seconds = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()

df["trip_duration_min"] = duration_seconds / 60

del duration_seconds

# duration filters before speed calculation
df = df[(df["trip_duration_min"] > 1) & (df["trip_duration_min"] <= 300)]

# speed
df["speed_mph"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

# speed filter
df = df[df["speed_mph"] <= 120]

print("Rows after duration/speed cleaning:", len(df))

Rows after duration/speed cleaning: 2657026


In [60]:
# 5. Duplicate removal
before_dupes = len(df)
df = df.drop_duplicates()
after_dupes = len(df)

print("Duplicates removed:", before_dupes - after_dupes)
print("Rows after duplicate removal:", len(df))

Duplicates removed: 0
Rows after duplicate removal: 2657026


In [61]:
# 6. Post-cleaning checks
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_after"))
display(df["passenger_count"].value_counts(dropna=False).sort_index().to_frame("count_after"))
display(df["trip_distance"].describe().to_frame("trip_distance_after"))
display(df["fare_amount"].describe().to_frame("fare_amount_after"))
display(df["trip_duration_min"].describe().to_frame("trip_duration_min_after"))
display(df["speed_mph"].describe().to_frame("speed_mph_after"))

,missing_after
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_distance,0
passenger_count,0
fare_amount,0
trip_duration_min,0
speed_mph,0


,count_after
passenger_count,
1.0,2100817
2.0,387974
3.0,87274
4.0,48333
5.0,32628


,trip_distance_after
count,2.657026e+06
mean,3.249257e+00
std,4.242640e+00
min,2.100000e-01
25%,1.020000e+00
50%,1.700000e+00
75%,3.110000e+00
max,8.000000e+01


,fare_amount_after
count,2.657026e+06
mean,1.811813e+01
std,1.583749e+01
min,1.000000e-02
25%,8.600000e+00
50%,1.280000e+01
75%,1.980000e+01
max,6.500000e+02


,trip_duration_min_after
count,2.657026e+06
mean,1.493590e+01
std,1.186802e+01
min,1.016667e+00
25%,7.266667e+00
50%,1.165000e+01
75%,1.863333e+01
max,2.992833e+02


,speed_mph_after
count,2.657026e+06
mean,1.144658e+01
std,6.599162e+00
min,9.006928e-02
25%,7.381295e+00
50%,9.637795e+00
75%,1.303866e+01
max,1.146951e+02


In [62]:
import pandas as pd


# CREATE YELLOW TAXI DATASETS FOR 2021-2025
YEARS = [2021, 2022, 2023, 2024, 2025]
MONTHS = range(1, 13)

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount"
]


# CLEANING FUNCTION
def clean_trips(df):

    # convert datetime
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    # drop missing essentials
    df = df.dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID"
    ])

    # validity filters
    df = df[df["fare_amount"] > 0]
    df = df[df["trip_distance"].between(0.2, 120)]
    df = df[df["PULocationID"].between(1, 263)]
    df = df[df["DOLocationID"].between(1, 263)]

    df = df[df["passenger_count"].between(1, 5)]

    # duration
    duration = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df["trip_duration_min"] = duration

    df = df[df["trip_duration_min"].between(1, 300)]

    # speed
    df["speed_mph"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

    df = df[df["speed_mph"] <= 120]

    # remove duplicates
    df = df.drop_duplicates()

    return df


# PROCESS EACH YEAR

for year in YEARS:

    print(f"\nProcessing {year}")

    monthly_frames = []


    # LOAD MONTHLY FILES

    for month in MONTHS:

        url = f"{BASE_URL}/yellow_tripdata_{year}-{month:02d}.parquet"

        try:

            df = pd.read_parquet(url, columns=COLUMNS)

            df = clean_trips(df)

            monthly_frames.append(df)

            print(f"{year}-{month:02d} loaded:", len(df))

        except Exception as e:

            print(f"{year}-{month:02d} skipped:", e)

    # combine full year
    df_year = pd.concat(monthly_frames, ignore_index=True)

    print("Total trips after cleaning:", len(df_year))

    # CREATE PICKUP HOUR

    df_year["pickup_hour"] = df_year["tpep_pickup_datetime"].dt.floor("h")

    # OBSERVED HOURLY DEMAND

    hourly_base = df_year[["PULocationID", "pickup_hour"]].copy()

    hourly_base["PULocationID"] = hourly_base["PULocationID"].astype("int16")

    observed = (
        hourly_base.value_counts(sort=False)
        .rename("trip_count")
        .reset_index()
    )

    # FULL HOURLY GRID
    full_hours = pd.date_range(
        start=f"{year}-01-01 00:00:00",
        end=f"{year}-12-31 23:00:00",
        freq="h"
    )

    zones = pd.DataFrame(
        {"PULocationID": range(1, 264)},
        dtype="int16"
    )

    full_grid = (
        zones.assign(key=1)
        .merge(
            pd.DataFrame({"pickup_hour": full_hours}).assign(key=1),
            on="key"
        )
        .drop("key", axis=1)
    )

    # MERGE DEMAND

    yearly_panel = full_grid.merge(
        observed,
        on=["PULocationID", "pickup_hour"],
        how="left"
    )

    yearly_panel["trip_count"] = yearly_panel["trip_count"].fillna(0).astype("int32")

    # TIME FEATURES
    yearly_panel["date"] = yearly_panel["pickup_hour"].dt.normalize()
    yearly_panel["year"] = yearly_panel["pickup_hour"].dt.year
    yearly_panel["month"] = yearly_panel["pickup_hour"].dt.month
    yearly_panel["day"] = yearly_panel["pickup_hour"].dt.day
    yearly_panel["hour"] = yearly_panel["pickup_hour"].dt.hour

    yearly_panel["day_of_week"] = yearly_panel["pickup_hour"].dt.dayofweek

    yearly_panel["is_weekend"] = yearly_panel["day_of_week"] >= 5

    yearly_panel["is_rush_hour"] = yearly_panel["hour"].isin([7,8,9,17,18,19])

    yearly_panel = yearly_panel.sort_values(
        ["PULocationID","pickup_hour"]
    ).reset_index(drop=True)

    print("Final panel shape:", yearly_panel.shape)


    # SAVE
    yearly_panel.to_parquet(
        f"yellow_hourly_demand_full_{year}.parquet",
        index=False
    )

    print(f"{year} saved")


Processing 2021
2021-01 loaded: 1173023
2021-02 loaded: 1179979
2021-03 loaded: 1667573
2021-04 loaded: 1896606
2021-05 loaded: 2215192
2021-06 loaded: 2515384
2021-07 loaded: 2490819
2021-08 loaded: 2452191
2021-09 loaded: 2604717
2021-10 loaded: 3092398
2021-11 loaded: 3108945
2021-12 loaded: 2887486
Total trips after cleaning: 27284313
Final panel shape: (2303880, 11)
2021 saved

Processing 2022
2022-01 loaded: 2225092
2022-02 loaded: 2679881
2022-03 loaded: 3266607
2022-04 loaded: 3240584
2022-05 loaded: 3212069
2022-06 loaded: 3178742
2022-07 loaded: 2842544
2022-08 loaded: 2830715
2022-09 loaded: 2835362
2022-10 loaded: 3280095
2022-11 loaded: 2909733
2022-12 loaded: 3023154
Total trips after cleaning: 35524578
Final panel shape: (2303880, 11)
2022 saved

Processing 2023
2023-01 loaded: 2789679
2023-02 loaded: 2645924
2023-03 loaded: 3099317
2023-04 loaded: 2987245
2023-05 loaded: 3184429
2023-06 loaded: 2990158
2023-07 loaded: 2626882
2023-08 loaded: 2549986
2023-09 loaded: 252

In [66]:
# check dataset has been created properly
df = pd.read_parquet("yellow_hourly_demand_full_2024.parquet")
(df["trip_count"] == 0).mean()

np.float64(0.6288741368682776)

In [67]:
df["trip_count"].max()

np.int32(988)

In [68]:
df = pd.read_parquet("yellow_hourly_demand_full_2024.parquet")

df["pickup_hour"].nunique()

8784

In [69]:
df.sort_values("trip_count", ascending=False).head(20)

,PULocationID,pickup_hour,trip_count,date,year,month,day,hour,day_of_week,is_weekend,is_rush_hour
692521,79,2024-11-03 01:00:00,988,2024-11-03,2024,11,3,1,6,True,False
2081490,237,2024-12-18 18:00:00,745,2024-12-18,2024,12,18,18,2,False,True
692497,79,2024-11-02 01:00:00,718,2024-11-02,2024,11,2,1,5,True,False
1406873,161,2024-02-29 17:00:00,712,2024-02-29,2024,2,29,17,3,False,True
1246869,142,2024-12-12 21:00:00,711,2024-12-12,2024,12,12,21,3,False,False
692353,79,2024-10-27 01:00:00,705,2024-10-27,2024,10,27,1,6,True,False
1406010,161,2024-01-24 18:00:00,697,2024-01-24,2024,1,24,18,2,False,True
1413090,161,2024-11-14 18:00:00,690,2024-11-14,2024,11,14,18,3,False,True
2081346,237,2024-12-12 18:00:00,684,2024-12-12,2024,12,12,18,3,False,True
1241518,142,2024-05-03 22:00:00,682,2024-05-03,2024,5,3,22,4,False,False


In [70]:
zone_totals = df.groupby("PULocationID")["trip_count"].sum()

zone_totals.sort_values().head(10)

PULocationID
103    0
5      0
104    0
99     1
110    1
84     2
204    2
187    2
44     4
105    4
Name: trip_count, dtype: int32

In [ ]:
# CREATE FHV TAXI DATASETS 2021-2026